## Search Evaluation

In [13]:
import json
from tqdm.auto import tqdm

import pandas as pd
from minsearch import AppendableIndex

In [4]:
with open("documents-with-ids.json", "rt") as f_in:
    documents = json.load(f_in)

In [5]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [7]:
def search(query, course="data-engineering-zoomcamp"):
    boost = {"question": 3.0, "section": 0.5}

    results = index.search(
        query=query,
        filter_dict={"course": course},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )
    return results

In [8]:
search("how to install kafka?")

[{'text': 'confluent-kafka: `pip install confluent-kafka` or `conda install conda-forge::python-confluent-kafka`\nfastavro: pip install fastavro\nAbhirup Ghosh\nCan install Faust Library for Module 6 Python Version due to dependency conflicts?\nThe Faust repository and library is no longer maintained - https://github.com/robinhood/faust\nIf you do not know Java, you now have the option to follow the Python Videos 6.13 & 6.14 here https://www.youtube.com/watch?v=BgAlVknDFlQ&list=PL3MmuxUbc_hJed7dXYoJw8DoCuVHhGEQb&index=80  and follow the RedPanda Python version here https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/06-streaming/python/redpanda_example - NOTE: I highly recommend watching the Java videos to understand the concept of streaming but you can skip the coding parts - all will become clear when you get to the Python videos and RedPanda files.',
  'section': 'Module 6: streaming with kafka',
  'question': 'Python Kafka: Installing dependencies for python3 06-st

In [10]:
df_ground_truth = pd.read_csv("ground-truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [12]:
ground_truth[1]

{'question': 'How can I get the course schedule?',
 'course': 'data-engineering-zoomcamp',
 'document': 'c02e79ef'}

In [14]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q["document"]
    results = search(query=q["question"], course=q["course"])
    relevance = [d["id"] == doc_id for d in results]
    relevance_total.append(relevance)

  0%|          | 0/4627 [00:00<?, ?it/s]

In [18]:
relevance_total[:10]

[[False, True, False, False, False],
 [False, False, False, False, False],
 [False, False, False, True, False],
 [False, False, False, False, True],
 [False, False, True, False, False],
 [True, False, False, False, False],
 [True, False, False, False, False],
 [True, False, False, False, False],
 [False, False, False, False, False],
 [True, False, False, False, False]]

### Hit-rate & MRR

In [16]:
def hit_rate(relevance_total):
    cnt = 0
    for doc in relevance_total:
        if True in doc:
            cnt += 1
    return cnt / len(relevance_total)

In [21]:
def mrr(relevance_total):
    total_score = 0.
    for doc in relevance_total:
        for rank in range(len(doc)):
            if doc[rank] == True:
                total_score += 1 / (rank + 1)
    return total_score / len(relevance_total)

In [22]:
hit_rate(relevance_total), mrr(relevance_total)

(0.7910092932785823, 0.686776889273108)

In [26]:
def evaluate(ground_truth, search_fn):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q["document"]
        results = search_fn(q)
        relevance = [d["id"] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [27]:
evaluate(ground_truth, lambda q: search(q["question"], q["course"]))

  0%|          | 0/4627 [00:00<?, ?it/s]

{'hit_rate': 0.7910092932785823, 'mrr': 0.686776889273108}